In [1]:


import tkinter as tk
from tkinter import ttk, Canvas, messagebox
import random
import time
import threading
from collections import defaultdict
import copy

class BinPackingSolver:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Bin Packing Solver")
        self.root.geometry("1000x700")
        self.root.configure(bg='#0f0f0f')
        
        self.solving = False
        self.solution_history = []

        self.setup_styles()
        self.create_widgets()
        
    def setup_styles(self):
        style = ttk.Style()
        style.theme_use('clam')

        bg_dark = '#0f0f0f'
        bg_medium = '#1a1a1a'
        bg_light = '#2d2d2d'
        accent = '#00ff88'
        text_white = '#ffffff'
        text_gray = '#cccccc'
        
        style.configure('Dark.TFrame', background=bg_dark)
        style.configure('Medium.TFrame', background=bg_medium, relief='raised', borderwidth=2)
        style.configure('Light.TFrame', background=bg_light, relief='raised', borderwidth=1)
        
        style.configure('Title.TLabel', 
                       font=('Helvetica', 20, 'bold'), 
                       background=bg_dark, 
                       foreground=accent)
        
        style.configure('Header.TLabel', 
                       font=('Helvetica', 12, 'bold'), 
                       background=bg_medium, 
                       foreground=text_white)
        
        style.configure('Info.TLabel', 
                       font=('Helvetica', 10), 
                       background=bg_medium, 
                       foreground=text_gray)
        
        style.configure('Accent.TButton', 
                       font=('Helvetica', 11, 'bold'), 
                       background=accent, 
                       foreground='black')
        
        style.configure('Dark.TButton', 
                       font=('Helvetica', 10), 
                       background=bg_light, 
                       foreground=text_white)
        
        style.configure('Modern.TEntry', 
                       font=('Helvetica', 10), 
                       fieldbackground=bg_light, 
                       foreground=text_white)
        
        style.configure('Modern.TCombobox', 
                       font=('Helvetica', 10), 
                       fieldbackground=bg_light, 
                       foreground=text_white)
        
    def create_widgets(self):
        
        main_frame = ttk.Frame(self.root, style='Dark.TFrame')
        main_frame.pack(fill='both', expand=True, padx=10, pady=10)
        
        
        title_label = ttk.Label(main_frame, text="Bin Packing Solver", style='Title.TLabel')
        title_label.pack(pady=(0, 20))
        
        
        content_frame = ttk.Frame(main_frame, style='Dark.TFrame')
        content_frame.pack(fill='both', expand=True)
        
        
        left_panel = ttk.Frame(content_frame, style='Medium.TFrame')
        left_panel.pack(side='left', fill='y', padx=(0, 10), pady=0)
        left_panel.configure(width=300)
        left_panel.pack_propagate(False)
        
        
        right_panel = ttk.Frame(content_frame, style='Medium.TFrame')
        right_panel.pack(side='right', fill='both', expand=True)
        
        self.create_control_panel(left_panel)
        self.create_visualization_panel(right_panel)
        
    def create_control_panel(self, parent):
        
        input_frame = ttk.Frame(parent, style='Light.TFrame')
        input_frame.pack(fill='x', padx=10, pady=10)
        
        ttk.Label(input_frame, text="Problem Configuration", style='Header.TLabel').pack(pady=(10, 15))
        
        
        ttk.Label(input_frame, text="Item Sizes (comma-separated):", style='Info.TLabel').pack(anchor='w', padx=10)
        self.item_sizes_entry = ttk.Entry(input_frame, style='Modern.TEntry', width=35)
        self.item_sizes_entry.pack(fill='x', padx=10, pady=(5, 10))
        self.item_sizes_entry.insert(0, "50,20,50,80")
        
        
        ttk.Label(input_frame, text="Bin Capacity:", style='Info.TLabel').pack(anchor='w', padx=10)
        self.bin_size_entry = ttk.Entry(input_frame, style='Modern.TEntry', width=35)
        self.bin_size_entry.pack(fill='x', padx=10, pady=(5, 10))
        self.bin_size_entry.insert(0, "100")
        
        
        algo_frame = ttk.Frame(parent, style='Light.TFrame')
        algo_frame.pack(fill='x', padx=10, pady=10)
        
        ttk.Label(algo_frame, text="Algorithm Selection", style='Header.TLabel').pack(pady=(10, 15))
        
        self.algorithm_var = tk.StringVar(value="First Fit")
        algorithms = ["First Fit", "Next Fit", "Genetic Algorithm", "Backtracking"]
        
        self.algorithm_combo = ttk.Combobox(algo_frame, textvariable=self.algorithm_var,
                                          values=algorithms, state="readonly", 
                                          style='Modern.TCombobox', width=32)
        self.algorithm_combo.pack(fill='x', padx=10, pady=5)
        
        
        control_frame = ttk.Frame(parent, style='Light.TFrame')
        control_frame.pack(fill='x', padx=10, pady=10)
        
        ttk.Label(control_frame, text="Controls", style='Header.TLabel').pack(pady=(10, 15))
        
        button_frame = ttk.Frame(control_frame, style='Light.TFrame')
        button_frame.pack(fill='x', padx=10, pady=10)
        
        self.solve_button = ttk.Button(button_frame, text="Solve", 
                                     style='Accent.TButton', command=self.solve_async)
        self.solve_button.pack(fill='x', pady=2)
        
        self.compare_button = ttk.Button(button_frame, text="Compare All", 
                                       style='Dark.TButton', command=self.compare_algorithms)
        self.compare_button.pack(fill='x', pady=2)
        
        self.reset_button = ttk.Button(button_frame, text="Reset", 
                                     style='Dark.TButton', command=self.reset_gui)
        self.reset_button.pack(fill='x', pady=2)
        
        
        stats_frame = ttk.Frame(parent, style='Light.TFrame')
        stats_frame.pack(fill='both', expand=True, padx=10, pady=10)
        
        ttk.Label(stats_frame, text="Statistics", style='Header.TLabel').pack(pady=(10, 15))
        
        self.stats_text = tk.Text(stats_frame, height=15, width=35, bg='#1a1a1a', fg='#ffffff',
                                 font=('Courier', 9), relief='flat', borderwidth=0)
        self.stats_text.pack(fill='both', expand=True, padx=10, pady=10)
        
    def create_visualization_panel(self, parent):

        canvas_frame = ttk.Frame(parent, style='Light.TFrame')
        canvas_frame.pack(fill='both', expand=True, padx=10, pady=10)
        

        ttk.Label(canvas_frame, text="Bin Visualization", style='Header.TLabel').pack(pady=(10, 15))
        

        self.optimal_fit_label = ttk.Label(canvas_frame, 
                                        text="Optimal Fit: -", 
                                        style='Header.TLabel')
        self.optimal_fit_label.pack(pady=(0, 15))
        

        self.canvas_container = ttk.Frame(canvas_frame)
        self.canvas_container.pack(fill='both', expand=True, padx=10, pady=10)
        
        self.canvas = Canvas(self.canvas_container, bg='#0f0f0f', highlightthickness=0)
        
        v_scrollbar = ttk.Scrollbar(self.canvas_container, orient='vertical', command=self.canvas.yview)
        v_scrollbar.pack(side='right', fill='y')
        
        h_scrollbar = ttk.Scrollbar(self.canvas_container, orient='horizontal', command=self.canvas.xview)
        h_scrollbar.pack(side='bottom', fill='x')
        
        self.canvas.configure(yscrollcommand=v_scrollbar.set, xscrollcommand=h_scrollbar.set)
        self.canvas.pack(side='left', fill='both', expand=True)
        
        self.canvas.bind("<MouseWheel>", self.on_mousewheel)
        self.canvas.bind("<Button-4>", self.on_mousewheel)
        self.canvas.bind("<Button-5>", self.on_mousewheel)
        
        history_frame = ttk.Frame(parent, style='Light.TFrame')
        history_frame.pack(fill='x', padx=10, pady=10)
        
        ttk.Label(history_frame, text="Solution History", style='Header.TLabel').pack(pady=(10, 5))
        
        self.history_tree = ttk.Treeview(history_frame, columns=('Algorithm', 'Bins', 'Efficiency', 'Time'), 
                                        show='headings', height=6)
        self.history_tree.heading('Algorithm', text='Algorithm')
        self.history_tree.heading('Bins', text='Bins Used')
        self.history_tree.heading('Efficiency', text='Efficiency %')
        self.history_tree.heading('Time', text='Time (ms)')
        
        self.history_tree.column('Algorithm', width=150)
        self.history_tree.column('Bins', width=80)
        self.history_tree.column('Efficiency', width=80)
        self.history_tree.column('Time', width=80)
        
        self.history_tree.pack(fill='x', padx=10, pady=10)
        
    def on_mousewheel(self, event):
        """Handle mouse wheel scrolling"""
        if event.delta:
            self.canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")
        elif event.num == 4:
            self.canvas.yview_scroll(-1, "units")
        elif event.num == 5:
            self.canvas.yview_scroll(1, "units")
        
    def solve_async(self):
        if self.solving:
            return
        
        thread = threading.Thread(target=self.solve_bin_packing)
        thread.daemon = True
        thread.start()
        
    def solve_bin_packing(self):
        try:
            self.solving = True
            self.solve_button.configure(text="Solving...")
            
            item_sizes = list(map(int, self.item_sizes_entry.get().split(',')))
            bin_size = int(self.bin_size_entry.get())
            algorithm = self.algorithm_var.get()
            
            if not item_sizes or bin_size <= 0:
                raise ValueError("Invalid input: Please provide valid item sizes and bin capacity")
            
            if any(item > bin_size for item in item_sizes):
                raise ValueError("Error: Some items are larger than bin capacity")
            
            start_time = time.perf_counter()
            solution = self.solve_with_algorithm(item_sizes, bin_size, algorithm)
            end_time = time.perf_counter()
            
            execution_time = (end_time - start_time) * 1000  
            
            total_space = len(solution) * bin_size
            used_space = sum(sum(bin_items) for bin_items in solution)
            efficiency = (used_space / total_space) * 100 if total_space > 0 else 0
            
            self.update_statistics(solution, execution_time, efficiency, algorithm)
            
            self.display_solution(solution)
            
            self.solution_history.append({
                'algorithm': algorithm,
                'bins': len(solution),
                'efficiency': efficiency,
                'time': execution_time,
                'solution': solution
            })
            
            self.update_history_tree()
            
        except Exception as e:
            messagebox.showerror("Error", f"An error occurred: {str(e)}")
        finally:
            self.solving = False
            self.solve_button.configure(text="Solve")
            self.solve_button.configure(text="Solve")
            
    def solve_with_algorithm(self, items, bin_size, algorithm):
        if algorithm == "First Fit":
            return self.first_fit(items, bin_size)
        elif algorithm == "Next Fit":
            return self.next_fit(items, bin_size)
        elif algorithm == "Genetic Algorithm":
            return self.genetic_algorithm(items, bin_size)
        elif algorithm == "Backtracking":
            return self.backtracking(items, bin_size)
        else:
            return self.first_fit(items, bin_size)
    
    def first_fit(self, items, bin_size):
        """First Fit algorithm - places each item in the first bin that can fit it"""
        bins = []
        
        for item in items:
            placed = False
            for bin_items in bins:
                if sum(bin_items) + item <= bin_size:
                    bin_items.append(item)
                    placed = True
                    break
            
            if not placed:
                bins.append([item])
        
        return bins
    
    def next_fit(self, items, bin_size):
        """Next Fit algorithm - places each item in the current bin, opens new one if needed"""
        bins = []
        current_bin = []
        
        for item in items:
            if sum(current_bin) + item <= bin_size:
                current_bin.append(item)
            else:
                bins.append(current_bin)
                current_bin = [item]
        
        if current_bin:  
            bins.append(current_bin)
        
        return bins
    
    def genetic_algorithm(self, items, bin_size, population_size=100, generations=200):
        """Genetic Algorithm for bin packing"""
        if not items:
            return []
        
        sorted_items = sorted(enumerate(items), key=lambda x: x[1], reverse=True)
        
        def create_individual():
            """Create a random valid individual"""
            individual = [0] * len(items)
            bins = []
            
            for i, item in enumerate(items):
                placed = False
                for bin_id, bin_items in enumerate(bins):
                    if sum(bin_items) + item <= bin_size:
                        bin_items.append(item)
                        individual[i] = bin_id
                        placed = True
                        break
                
                if not placed:
                    bins.append([item])
                    individual[i] = len(bins) - 1
            
            
            if random.random() < 0.3:  
                for i in range(len(individual)):
                    if random.random() < 0.1:  
                        individual[i] = random.randint(0, max(individual) + 1)
            
            return individual
        
        def fitness(individual):
            """Calculate fitness (lower is better)"""
            bins = defaultdict(list)
            
            for i, bin_id in enumerate(individual):
                bins[bin_id].append(items[i])
            
            penalty = 0
            valid_bins = 0
            
            for bin_items in bins.values():
                bin_sum = sum(bin_items)
                if bin_sum > bin_size:
                    penalty += (bin_sum - bin_size) * 100  
                else:
                    valid_bins += 1
            
            total_bins = len(bins)
            wasted_space = sum(max(0, bin_size - sum(bin_items)) for bin_items in bins.values())
            
            return total_bins * 1000 + penalty + wasted_space
        
        def crossover(parent1, parent2):
            """Single point crossover"""
            if len(parent1) != len(parent2):
                return parent1[:], parent2[:]
            
            point = random.randint(1, len(parent1) - 1)
            child1 = parent1[:point] + parent2[point:]
            child2 = parent2[:point] + parent1[point:]
            
            return child1, child2
        
        def mutate(individual):
            """Mutate by changing bin assignments"""
            individual = individual[:]
            
            for i in range(len(individual)):
                if random.random() < 0.05:  
                    
                    max_bin = max(individual) if individual else 0
                    new_bin = random.randint(0, max_bin + 2)
                    individual[i] = new_bin
            
            return individual
        
        def repair(individual):
            """Repair invalid solutions using First Fit"""
            bins = defaultdict(list)
            for i, bin_id in enumerate(individual):
                bins[bin_id].append((i, items[i]))
            
           
            valid_bins = []
            items_to_repack = []
            
            for bin_items in bins.values():
                bin_sum = sum(item for _, item in bin_items)
                if bin_sum <= bin_size:
                    valid_bins.append([item for _, item in bin_items])
                else:
                    items_to_repack.extend(bin_items)
            
            
            new_individual = [0] * len(items)
            
            
            for bin_id, bin_items in enumerate(valid_bins):
                for item_val in bin_items:
                    for i, orig_item in enumerate(items):
                        if orig_item == item_val and new_individual[i] == 0:
                            new_individual[i] = bin_id
                            break
            
            
            next_bin_id = len(valid_bins)
            for item_idx, item_val in items_to_repack:
                placed = False
                
               
                for bin_id, bin_items in enumerate(valid_bins):
                    if sum(bin_items) + item_val <= bin_size:
                        bin_items.append(item_val)
                        new_individual[item_idx] = bin_id
                        placed = True
                        break
                
                
                if not placed:
                    valid_bins.append([item_val])
                    new_individual[item_idx] = next_bin_id
                    next_bin_id += 1
            
            return new_individual
        
        
        population = [repair(create_individual()) for _ in range(population_size)]
        
        best_fitness = float('inf')
        stagnation_count = 0
        
        for generation in range(generations):
            
            fitness_scores = [(fitness(ind), ind) for ind in population]
            fitness_scores.sort()
            
            current_best = fitness_scores[0][0]
            if current_best < best_fitness:
                best_fitness = current_best
                stagnation_count = 0
            else:
                stagnation_count += 1
            
           
            if stagnation_count > 50:
                break
            
            
            elite_size = population_size // 4
            elite = [ind for _, ind in fitness_scores[:elite_size]]
            
            
            new_population = elite[:]
            
            while len(new_population) < population_size:
                
                tournament_size = 3
                parent1 = min(random.sample(population, tournament_size), key=fitness)
                parent2 = min(random.sample(population, tournament_size), key=fitness)
                
                
                child1, child2 = crossover(parent1, parent2)
                child1 = repair(mutate(child1))
                child2 = repair(mutate(child2))
                
                new_population.extend([child1, child2])
            
            population = new_population[:population_size]
        
        
        best_individual = min(population, key=fitness)
        bins = defaultdict(list)
        
        for i, bin_id in enumerate(best_individual):
            bins[bin_id].append(items[i])
        
        
        result = [bin_items for bin_items in bins.values() if bin_items]
        return result
    
    def backtracking(self, items, bin_size):
        """Backtracking algorithm with optimizations"""
        if not items:
            return []
        
        if len(items) > 12:
            return self.first_fit(items, bin_size)
        
        items_with_index = [(i, item) for i, item in enumerate(items)]
        items_with_index.sort(key=lambda x: x[1], reverse=True)
        sorted_items = [item for _, item in items_with_index]
        
        best_solution = None
        best_bins_count = len(items)  
        
        def lower_bound(remaining_items):
            """Calculate lower bound on number of bins needed"""
            if not remaining_items:
                return 0
            return (sum(remaining_items) + bin_size - 1) // bin_size
        
        def backtrack(index, bins):
            nonlocal best_solution, best_bins_count
            
            if index == len(sorted_items):
                if len(bins) < best_bins_count:
                    best_bins_count = len(bins)
                    best_solution = [bin[:] for bin in bins]
                return
            
            remaining_items = sorted_items[index:]
            if len(bins) + lower_bound(remaining_items) >= best_bins_count:
                return
            
            item = sorted_items[index]
            
            for i, bin_items in enumerate(bins):
                if sum(bin_items) + item <= bin_size:
                    bins[i].append(item)
                    backtrack(index + 1, bins)
                    bins[i].pop()
            
            if len(bins) + 1 < best_bins_count:
                bins.append([item])
                backtrack(index + 1, bins)
                bins.pop()
        
        backtrack(0, [])
        
        if best_solution is None:
            return self.first_fit(items, bin_size)
        
        return best_solution
    
    def calculate_optimal_fit(self, items, bin_size):
        """Calculate the theoretical minimum number of bins required"""
        if not items:
            return 0
        return (sum(items) + bin_size - 1) // bin_size  
    
    def display_solution(self, solution):
        self.canvas.delete("all")
    
        if not solution:
            return
        
        
        items = list(map(int, self.item_sizes_entry.get().split(',')))
        bin_size = int(self.bin_size_entry.get())
        optimal_bins = self.calculate_optimal_fit(items, bin_size)
        self.optimal_fit_label.config(text=f"Optimal Fit: {optimal_bins} bins")
            
        colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4', '#ffeaa7', '#fd79a8', 
                  '#a8e6cf', '#dcedc1', '#ffd3a5', '#fd9644', '#f093fb', '#f5576c']
        
        canvas_width = self.canvas.winfo_width()
        if canvas_width <= 1:
            canvas_width = 600
        
        bin_width = min(400, canvas_width - 100)
        visual_bin_width = int(bin_width * 1.009)  
        bin_height = 80
        padding = 20

        
        total_height = len(solution) * (bin_height + padding) + padding

        
        self.canvas.configure(scrollregion=(0, 0, 0, total_height))

        for i, bin_items in enumerate(solution):
            y = padding + i * (bin_height + padding)
            x = 50

            
            self.canvas.create_rectangle(x, y, x + visual_bin_width, y + bin_height - 20, 
                                       fill='#2d2d2d', outline='#00ff88', width=2)

            
            bin_capacity = int(self.bin_size_entry.get())
            used_space = sum(bin_items)
            efficiency = (used_space / bin_capacity) * 100

            self.canvas.create_text(x + 10, y - 10, 
                                  text=f"Bin {i+1}: {used_space}/{bin_capacity} ({efficiency:.1f}%)",
                                  fill='white', font=('Helvetica', 11, 'bold'), anchor='w')

            
            item_x = x + 10
            item_y = y + 10

            for j, item in enumerate(bin_items):
                if bin_capacity > 0:
                    item_width = max(20, (item / bin_capacity) * (bin_width - 20))
                else:
                    item_width = 20
                item_height = 40

                if item_x + item_width > x + visual_bin_width - 10:
                    item_x = x + 10
                    item_y += item_height + 5

                color = colors[j % len(colors)]

                self.canvas.create_rectangle(item_x, item_y, item_x + item_width, item_y + item_height,
                                           fill=color, outline='white', width=1)

                if item_width > 25:  
                    self.canvas.create_text(item_x + item_width/2, item_y + item_height/2,
                                          text=str(item), fill='white', 
                                          font=('Helvetica', 9, 'bold'))

                item_x += item_width + 2
    
    def update_statistics(self, solution, execution_time, efficiency, algorithm):
        self.stats_text.delete(1.0, tk.END)
        
        total_items = len(self.item_sizes_entry.get().split(','))
        items = list(map(int, self.item_sizes_entry.get().split(',')))
        bin_size = int(self.bin_size_entry.get())
        optimal_bins = self.calculate_optimal_fit(items, bin_size)
        
        stats = f"""Algorithm: {algorithm}
Bins Used: {len(solution)}
Optimal Bins: {optimal_bins}
Total Items: {total_items}
Execution Time: {execution_time:.3f} ms
Space Efficiency: {efficiency:.1f}%

Bin Details:
"""  
        bin_capacity = int(self.bin_size_entry.get())
        for i, bin_items in enumerate(solution):
            used_space = sum(bin_items)
            bin_efficiency = (used_space / bin_capacity) * 100
            stats += f"Bin {i+1}: {bin_items} = {used_space}/{bin_capacity} ({bin_efficiency:.1f}%)\n"
        
        self.stats_text.insert(tk.END, stats)
        self.stats_text.see(tk.END)
    
    def update_history_tree(self):
        for item in self.history_tree.get_children():
            self.history_tree.delete(item)
        
        for entry in self.solution_history:
            self.history_tree.insert('', 'end', values=(
                entry['algorithm'],
                entry['bins'],
                f"{entry['efficiency']:.1f}%",
                f"{entry['time']:.3f}"
            ))
    
    def compare_algorithms(self):
        try:
            item_sizes = list(map(int, self.item_sizes_entry.get().split(',')))
            bin_size = int(self.bin_size_entry.get())
            
            if not item_sizes or bin_size <= 0:
                raise ValueError("Invalid input: Please provide valid item sizes and bin capacity")
            
            if any(item > bin_size for item in item_sizes):
                raise ValueError("Error: Some items are larger than bin capacity")
            
            algorithms = ["First Fit", "Next Fit", "Genetic Algorithm", "Backtracking"]
            
            results = {}
            
            self.compare_button.configure(text="Comparing...")
            self.root.update()
            
            for algorithm in algorithms:
                try:
                    start_time = time.perf_counter()
                    solution = self.solve_with_algorithm(item_sizes, bin_size, algorithm)
                    end_time = time.perf_counter()
                    
                    execution_time = (end_time - start_time) * 1000
                    total_space = len(solution) * bin_size
                    used_space = sum(sum(bin_items) for bin_items in solution)
                    efficiency = (used_space / total_space) * 100 if total_space > 0 else 0
                    
                    results[algorithm] = {
                        'bins': len(solution),
                        'time': execution_time,
                        'efficiency': efficiency,
                        'solution': solution
                    }
                    
                    self.solution_history.append({
                        'algorithm': algorithm,
                        'bins': len(solution),
                        'efficiency': efficiency,
                        'time': execution_time,
                        'solution': solution
                    })
                    
                except Exception as e:
                    print(f"Error with {algorithm}: {e}")
                    solution = self.first_fit(item_sizes, bin_size)
                    results[algorithm] = {
                        'bins': len(solution),
                        'time': 0,
                        'efficiency': 0,
                        'solution': solution
                    }
            
            self.update_history_tree()
            
            best_algorithm = min(results.keys(), key=lambda x: results[x]['bins'])
            best_solution = results[best_algorithm]['solution']
            
            self.display_solution(best_solution)
            self.update_statistics(best_solution, results[best_algorithm]['time'], 
                                 results[best_algorithm]['efficiency'], best_algorithm)
            
            comparison_text = "Algorithm Comparison Results:\n\n"
            for algo in algorithms:
                if algo in results:
                    comparison_text += f"{algo}:\n"
                    comparison_text += f"  Bins: {results[algo]['bins']}\n"
                    comparison_text += f"  Efficiency: {results[algo]['efficiency']:.1f}%\n"
                    comparison_text += f"  Time: {results[algo]['time']:.3f}ms\n\n"
            
            comparison_text += f"Best Algorithm: {best_algorithm}"
            
            messagebox.showinfo("Comparison Complete", comparison_text)
            
        except Exception as e:
            messagebox.showerror("Error", f"An error occurred during comparison: {str(e)}")
        finally:
            self.compare_button.configure(text="Compare All")
    
    def reset_gui(self):
        self.canvas.delete("all")
        self.stats_text.delete(1.0, tk.END)
        self.solution_history.clear()
        
        for item in self.history_tree.get_children():
            self.history_tree.delete(item)
        
        self.item_sizes_entry.delete(0, tk.END)
        self.item_sizes_entry.insert(0, "50,20,50,80")
        
        self.bin_size_entry.delete(0, tk.END)
        self.bin_size_entry.insert(0, "100")
        
        self.algorithm_var.set("First Fit")
        
        self.solve_button.configure(text="Solve")
        self.compare_button.configure(text="Compare All")
        
        welcome_text = """Welcome to Bin Packing Solver!

Instructions:
1. Enter item sizes separated by commas
2. Set the bin capacity
3. Choose an algorithm
4. Click 'Solve' to find a solution
5. Use 'Compare All' to test all algorithms

Algorithms Available:
• First Fit: Quick greedy approach
• Best Fit: Minimizes wasted space
• Genetic Algorithm: Evolutionary optimization
• Backtracking: Exhaustive search (for small problems)

The visualization will show bins with items
colored by their placement order."""
        
        self.stats_text.insert(tk.END, welcome_text)
    
    def run(self):
        """Start the GUI application"""
        self.reset_gui()
        
        self.canvas.bind('<Configure>', self.on_canvas_resize)
        
        self.root.mainloop()
    
    def on_canvas_resize(self, event):
        """Handle canvas resize events"""
        if hasattr(self, 'solution_history') and self.solution_history:
            last_solution = self.solution_history[-1]['solution']
            self.display_solution(last_solution)

if __name__ == "__main__":
    app = BinPackingSolver()
    app.run()